In [3]:
def plot_switched_heads_heatmaps_uniform(switched_head_counts_per_map, map_order):
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # 1) Define a pretty name map here:
    pretty_names = {
        "proper_general_heads":      "General Heads",
        "proper_entity_heads":       "Entity Heads",
        "proper_relation_answer_heads":"Relation–Answer Heads",
        "proper_answer_specific_heads":"Answer-Specific Heads",
        # …add any others you have…
    }

    # 2) Build your DataFrame as before
    plot_data = []
    for map_type in map_order:
        counter = switched_head_counts_per_map[map_type]
        for (layer, head), count in counter.items():
            plot_data.append((map_type, layer, head, count))
    df = pd.DataFrame(plot_data, columns=["Map Type", "Layer", "Head", "Switches"])

    # 3) Global color range
    global_vmin, global_vmax = 0, df["Switches"].max()

    # 4) Subplots
    fig, axes = plt.subplots(2, 2, figsize=(12,12), sharex=True, sharey=True)
    axes = axes.flatten()

    for ax, map_type in zip(axes, map_order):
        data = df[df["Map Type"] == map_type]
        # Pivot into 32×32 matrix
        heatmap_data = (
            data
            .pivot(index="Layer", columns="Head", values="Switches")
            .reindex(index=range(32), columns=range(32), fill_value=0)
        )

        sns.heatmap(
            heatmap_data, ax=ax, cmap="Blues",
            vmin=global_vmin, vmax=global_vmax,
            cbar=True, annot=False
        )
        ax.invert_yaxis()
        sns.set_style("white")

        # 5) Use the pretty name if available, else fall back
        title = pretty_names.get(map_type, map_type.replace("_", " ").title())
        ax.set_title(f"Switched {title}", fontsize=14)

        ax.set_xlabel("Head Number", fontsize=12)
        ax.set_ylabel("Layer Number", fontsize=12)
        ax.tick_params(labelsize=10)
        ax.set_xticks(range(0,32,2))
        ax.set_yticks(range(0,32,2))

    # Hide any extra axes
    for ax in axes[len(map_order):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.savefig("final_figures2/switches_aggregated.pdf", dpi=500, bbox_inches='tight', format="pdf")
    plt.show()

In [1]:
from collections import Counter

# Define the correct order of map types for subplots
map_order = ["proper_general_heads", "proper_entity_heads", "proper_relation_answer_heads", "proper_answer_specific_heads"]

# Initialize a dictionary to store counters per map_type
switched_head_counts_per_map_LOC = {map_type: Counter() for map_type in map_order}

# Iterate through results to collect switched head counts per map_type
for map_type in results2:
    if map_type in switched_head_counts_per_map_LOC:  # Ensure we only process expected types
        for transition in results2[map_type]:
            switched_heads = results2[map_type][transition]["switched_indices"]
            for layer, head in switched_heads:
                switched_head_counts_per_map_LOC[map_type][(layer, head)] += 1

plot_switched_heads_heatmaps_uniform(switched_head_counts_per_map_LOC, map_order)

NameError: name 'results2' is not defined